In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import plotly.graph_objects as go

In [ ]:
with h5py.File('nexus_outputs.h5', 'r') as f:
    norm_dens = f['z_0.0']['density'][:]
    voids = f['z_0.0']['voids'][:]
    filaments = f['z_0.0']['filaments'][:]
    nodes = f['z_0.0']['nodes'][:]
    walls = f['z_0.0']['walls'][:]
    print(f"Suessfully Loaded Stuff")

norm_dens = np.transpose(norm_dens, (1, 2, 0))
voids = np.transpose(voids, (1, 2, 0))
filaments = np.transpose(filaments, (1, 2, 0))
nodes = np.transpose(nodes, (1, 2, 0))
walls = np.transpose(walls, (1, 2, 0))

In [ ]:
basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

coords_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000 #in cMpc/h
ids_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
slice_index = 0

z_min = slice_index * dl
z_max = (slice_index + 1) * dl
mask = (coords_0[:,2] > z_min) & (coords_0[:,2] < z_max)

x = coords_0[:, 0][mask]
y = coords_0[:, 1][mask]
z = coords_0[:, 2][mask]

fig = plt.figure(figsize=(12,4))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(np.log10(norm_dens[:,:,slice_index]),origin = "lower",cmap="inferno", extent = [0,35,0,35])
ax1.set_xlabel("cMpc/h")
ax1.set_ylabel("cMpc/h")
ax1.set_aspect('equal')
fig.colorbar(im1, ax = ax1, label = r"$log_{10}(1+\delta)$")

ax2 = fig.add_subplot(1,2,2)
counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
ax2.set_xlabel("cMpc/h")
ax2.set_ylabel("cMpc/h")
ax2.set_aspect('equal')
fig.colorbar(im2, ax = ax2, label = "Counts")

plt.tight_layout();

In [ ]:
bool_filament = filaments.astype(bool)
bool_wall = walls.astype(bool)
bool_node = nodes.astype(bool)
bool_void = voids.astype(bool)

grid_x = np.floor(coords_0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_0[:, 2] / dl).astype(int) % res

In [ ]:
in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_0[in_node_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_0[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_0[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_0[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")

In [ ]:
match_mask = np.isin(ids_0, node_particle_ids)
node_coords = coords_0[match_mask]

nx = node_coords[:, 0]
ny = node_coords[:, 1]
nz = node_coords[:, 2]

match_mask = np.isin(ids_0, filament_particle_ids)
filament_coords = coords_0[match_mask]

fx = filament_coords[:, 0]
fy = filament_coords[:, 1]
fz = filament_coords[:, 2]

match_mask = np.isin(ids_0, wall_particle_ids)
wall_coords = coords_0[match_mask]

wx = wall_coords[:, 0]
wy = wall_coords[:, 1]
wz = wall_coords[:, 2]

match_mask = np.isin(ids_0, void_particle_ids)
void_coords = coords_0[match_mask]

vx = void_coords[:, 0]
vy = void_coords[:, 1]
vz = void_coords[:, 2]

In [ ]:
fig = go.Figure()

# #fig.add_trace(go.Scatter3d(x=vx[::1000], y=vy[::1000], z=vz[::1000],
#     mode='markers', marker=dict(
#     size=1, 
#     color="blue",             
#     #colorscale='Viridis', 
#     opacity=0.25
#     ), 
#     name = "Void particle"))

#fig.add_trace(go.Scatter3d(x=wx[::500], y=wy[::500], z=wz[::500],
    # mode='markers', marker=dict(
    # size=1,  
    # color="orange",              
    # #colorscale='Viridis',  
    # opacity=0.5
    # ),
    # name = "Wall particle"))


fig.add_trace(go.Scatter3d(x=fx[::10], y=fy[::10], z=fz[::10],
    mode='markers', marker=dict(
    size=0.25,  
    color="green",
    #colorscale='Viridis',   # choose a colorscale
    opacity=1
    ), 
    name = "Filament particle"))

#fig.add_trace(go.Scatter3d(x=nx, y=ny, z=nz,
    #mode='markers', marker=dict(
    #size=1,  
    #color="red",          
    ##colorscale='Viridis',   
    #opacity=1
    #),
    #name = "Node particle"))


fig.show()

In [ ]:
# 1. Create a 3D histogram (bin the particles to get density)
# Choose the number of bins (e.g., 32x32x32). Higher = more resolution but slower.
bins = 128
counts, edges = np.histogramdd((fx, fy, fz), bins=bins)

# 2. Get the center coordinates of each bin
x_centers = (edges[0][:-1] + edges[0][1:]) / 2
y_centers = (edges[1][:-1] + edges[1][1:]) / 2
z_centers = (edges[2][:-1] + edges[2][1:]) / 2

X, Y, Z = np.meshgrid(x_centers, y_centers, z_centers, indexing='ij')

# 3. Take the log of the particle counts (adding 1 to avoid log(0))
log_density = np.log10(counts + 1)

# 4. Plot using Volume rendering
fig = go.Figure(data=go.Volume(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=log_density.flatten(), # The log of particle counts dictates color
    isomin=0.1,                  # Hide empty space (log10(1) = 0)
    isomax=np.max(log_density),
    opacity=0.1,                 # Make it see-through
    surface_count=15,            # Number of layers
    colorscale='Plasma'
))

fig.update_layout(scene=dict(bgcolor="black"), paper_bgcolor="black")
fig.show()